# 01 — Train one model / one strategy on Colab

**Prerequisite:** run `00_colab_setup.ipynb` once (it caches the dataset zip on Google Drive).

1. `Runtime > Change runtime type > T4 GPU`
2. Choose `MODEL` and `STRATEGY` in the next cell, then `Runtime > Run all`.

Results are written to Google Drive (`MyDrive/road-damage-tl/runs/`) and to MLflow
(database `MyDrive/road-damage-tl/mlflow.db`, artifacts in `MyDrive/road-damage-tl/mlartifacts/`),
so nothing is lost when the Colab session ends.

| MODEL | Owner | STRATEGY |
|---|---|---|
| `scratch_cnn` | Student 2 | (ignored — trained from scratch) |
| `alexnet` | Student 2 | `feature_extraction`, `partial_finetune`, `full_finetune` |
| `convnextv2` | Student 1 | idem |
| `vit_b16` | Student 3 | idem |
| `nfnet_f1` | Student 3 | idem (heavy: use `BATCH_SIZE=16`, `ACCUM_STEPS=2`) |

In [ ]:
MODEL = "scratch_cnn"          # scratch_cnn | alexnet | convnextv2 | nfnet_f1 | vit_b16
STRATEGY = "feature_extraction"  # feature_extraction | partial_finetune | full_finetune
EPOCHS = 30
BATCH_SIZE = 32
ACCUM_STEPS = 1                # effective batch = BATCH_SIZE x ACCUM_STEPS (keep it = 32)

REPO_URL = "https://github.com/Brahim-semlali/road-damage-tl.git"
DRIVE_ROOT = "/content/drive/MyDrive/road-damage-tl"
REPO_DIR = "/content/road-damage-tl"
RAW_DIR = "/content/data/raw"

## 1. Setup: Drive, code, dependencies, data

In [ ]:
import glob, os, subprocess
from google.colab import drive
drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log --oneline -1

!pip install -q timm albumentations mlflow

if not os.path.isdir(f"{RAW_DIR}/sih_road_dataset"):
    zips = sorted(glob.glob(f"{DRIVE_ROOT}/datasets/*.zip"))
    assert zips, "No dataset zip on Drive: run 00_colab_setup.ipynb first."
    os.makedirs(RAW_DIR, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", RAW_DIR], check=True)
print(sum(len(f) for _, _, f in os.walk(RAW_DIR)), "files in", RAW_DIR)

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Quick sanity check (2 batches, 1 epoch)

Catches errors in seconds before launching a long run.

In [ ]:
!python -m src.train --model {MODEL} --strategy {STRATEGY} --epochs 1 --max-batches 2 \
    --batch-size {BATCH_SIZE} --data-root {RAW_DIR} --out /content/debug_runs

## 3. Training

In [ ]:
# MLflow SQLite database: used on the local disk (SQLite is unreliable on the Drive file
# system) and copied back to Drive after the run.
import shutil
DB_LOCAL, DB_DRIVE = "/content/mlflow.db", f"{DRIVE_ROOT}/mlflow.db"
if os.path.exists(DB_DRIVE):
    shutil.copy(DB_DRIVE, DB_LOCAL)

!python -m src.train --model {MODEL} --strategy {STRATEGY} --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} --accum-steps {ACCUM_STEPS} \
    --data-root {RAW_DIR} --out "{DRIVE_ROOT}/runs" \
    --mlflow-uri "sqlite:///{DB_LOCAL}" --mlflow-artifacts "{DRIVE_ROOT}/mlartifacts"

if os.path.exists(DB_LOCAL):
    shutil.copy(DB_LOCAL, DB_DRIVE)

## 4. Results of this run

In [ ]:
import json
from IPython.display import Image, display
run = sorted(glob.glob(f"{DRIVE_ROOT}/runs/{MODEL}__*"), key=os.path.getmtime)[-1]
print("Run folder:", run)
assert os.path.exists(f"{run}/metrics.json"), \
    "Training did not finish: read the error at the end of the output of cell 3."
print(json.dumps(json.load(open(f"{run}/metrics.json")), indent=2))
display(Image(f"{run}/curves.png", width=900))
display(Image(f"{run}/confusion_matrix.png", width=450))

## 5. Summary of all runs so far

This table becomes the "Global results" table of the paper (Section IV.6).

In [ ]:
import pandas as pd
rows = []
for f in sorted(glob.glob(f"{DRIVE_ROOT}/runs/*/metrics.json")):
    m = json.load(open(f))
    name = os.path.basename(os.path.dirname(f)).split("__")
    rows.append({"model": name[0], "strategy": name[1], **m})
cols = ["model", "strategy", "test_accuracy", "test_precision_macro", "test_recall_macro",
        "test_f1_macro", "test_auc_macro_ovr", "params_total", "params_trainable",
        "train_time_s", "peak_gpu_mem_mb", "latency_ms", "best_epoch"]
pd.DataFrame(rows)[cols].round(4)